In [19]:
import sys
import os

sys.path.append(os.path.abspath('../src'))

from models import calcular_costo_negocio

In [26]:
import time
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

In [21]:
# =========================================================
# 1. CARGA DE DATOS DEL EXPERIMENTO
# =========================================================
# Ruta A (Manual)
X_train_manual_sm = joblib.load('../data/processed/X_train_manual_sm.pkl')
X_val_manual = joblib.load('../data/processed/X_val_manual.pkl')

# Ruta B (Full)
X_train_full_sm = joblib.load('../data/processed/X_train_full_sm.pkl')
X_val_full = joblib.load('../data/processed/X_val_full.pkl')

# Variables compartidas (Target)
y_train_sm = joblib.load('../data/processed/y_train_sm.pkl')
y_val = joblib.load('../data/processed/y_val.pkl')

print("✅ Datos cargados exitosamente en memoria. ¡Listos para modelar!")

✅ Datos cargados exitosamente en memoria. ¡Listos para modelar!


In [22]:
# =========================================================================
# 3. CONFIGURACIÓN DEL GRAN EXPERIMENTO (8 MODELOS)
# =========================================================================

experimentos = [
    # -----------------------------------------------------------------
    # RUTA A: PURGA MANUAL (Se eliminó ruido y multicolinealidad)
    # -----------------------------------------------------------------
    {
        'nombre': 'Regresión Logística (Ruta A - Manual)',
        'ruta': 'manual',
        'modelo': LogisticRegression(random_state=42, max_iter=1000),
        'grid': {'C': [0.01, 0.1, 1, 10, 100]}
    },
    {
        'nombre': 'SVM (Ruta A - Manual)',
        'ruta': 'manual',
        'modelo': SVC(random_state=42, kernel='rbf'),
        'grid': {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto']}
    },
    {
        'nombre': 'Random Forest (Ruta A - Manual)',
        'ruta': 'manual',
        'modelo': RandomForestClassifier(random_state=42),
        'grid': {'n_estimators': [100, 200], 'max_depth': [5, 10, None], 'min_samples_split': [2, 5]}
    },
    {
        'nombre': 'XGBoost (Ruta A - Manual)',
        'ruta': 'manual',
        'modelo': XGBClassifier(random_state=42, eval_metric='logloss'),
        'grid': {'n_estimators': [100, 200], 'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.1]}
    },
    
    # -----------------------------------------------------------------
    # RUTA B: FULL DATA (Regularización algorítmica L1 / L2)
    # -----------------------------------------------------------------
    {
        'nombre': 'Regresión Logística (Ruta B - ElasticNet)',
        'ruta': 'full',
        # Usamos solver 'saga' porque es el único que soporta ElasticNet (L1 + L2 simultáneo)
        'modelo': LogisticRegression(random_state=42, solver='saga', penalty='elasticnet', max_iter=4000),
        'grid': {'C': [0.01, 0.1, 1, 10], 'l1_ratio': [0.2, 0.5, 0.8]} 
    },
    {
        'nombre': 'SVM (Ruta B - Fuerte L2)',
        'ruta': 'full',
        'modelo': SVC(random_state=42, kernel='rbf'),
        # Probamos valores de C más pequeños para forzar una regularización L2 agresiva contra la colinealidad
        'grid': {'C': [0.001, 0.01, 0.1, 1]} 
    },
    {
        'nombre': 'Random Forest (Ruta B - Full)',
        'ruta': 'full',
        'modelo': RandomForestClassifier(random_state=42),
        'grid': {'n_estimators': [100, 200], 'max_depth': [5, 10, None], 'min_samples_split': [2, 5]}
    },
    {
        'nombre': 'XGBoost (Ruta B - Regularizado L1/L2)',
        'ruta': 'full',
        'modelo': XGBClassifier(random_state=42, eval_metric='logloss'),
        'grid': {
            'n_estimators': [100, 200],
            'max_depth': [3, 5, 7],
            'reg_alpha': [0.1, 1.0, 5.0],   # Penalización L1 (Lasso)
            'reg_lambda': [0.1, 1.0, 5.0]  # Penalización L2 (Ridge)
        }
    }
]

In [23]:
# =========================================================================
# 4. EJECUCIÓN DEL BUCLE DE ENTRENAMIENTO Y EVALUACIÓN
# =========================================================================

resultados_lista = []
modelos_entrenados = {}

print("====== INICIANDO EL TORNEO DE ALGORITMOS ======\n")

for exp in experimentos:
    print(f"🤖 Entrenando: {exp['nombre']}...")
    tiempo_inicio = time.time()
    
    # 1. Selección dinámica del Dataset según la Ruta de la hipótesis
    if exp['ruta'] == 'manual':
        X_train_act = X_train_manual_sm
        X_val_act = X_val_manual
    else:
        X_train_act = X_train_full_sm
        X_val_act = X_val_full
        
    # 2. Configuración de la búsqueda de hiperparámetros con Validación Cruzada (CV=5)
    # Buscamos optimizar el F1-Score para cumplir con la rúbrica académica
    grid_search = GridSearchCV(
        estimator=exp['modelo'],
        param_grid=exp['grid'],
        scoring='f1',
        cv=5,
        n_jobs=-1,
        verbose=0
    )
    
    # 3. Ajuste del modelo
    grid_search.fit(X_train_act, y_train_sm)
    mejor_modelo = grid_search.best_estimator_
    
    # 4. Predicción en el conjunto de VALIDACIÓN (Data Real, NO sobremuestreada)
    y_pred = mejor_modelo.predict(X_val_act)
    
    # 5. Cálculo de métricas técnicas y de negocio
    f1_val = f1_score(y_val, y_pred)
    metricas_negocio = calcular_costo_negocio(y_val, y_pred)
    
    tiempo_total = time.time() - tiempo_inicio
    print(f"   ✓ ¡Completado! [Mejor F1 en CV: {grid_search.best_score_:.4f}] -> [F1 en Validación: {f1_val:.4f}]")
    print(f"   ✓ Costo Total para la Empresa: ${metricas_negocio['COSTO TOTAL ($)']:,.2f} | Tiempo: {tiempo_total:.1f}s\n")
    
    # 6. Almacenar resultados para el reporte analítico final
    resultados_lista.append({
        'Modelo / Algoritmo': exp['nombre'],
        'Estrategia Datos': exp['ruta'].upper(),
        'F1-Score (Validación)': round(f1_val, 4),
        'Falsos Positivos (Descuentos Inútiles)': metricas_negocio['Falsos Positivos (Gastos Innecesarios)'],
        'Falsos Negativos (Fugas No Detectadas)': metricas_negocio['Falsos Negativos (Fugas No Detectadas)'],
        'PÉRDIDA ECONÓMICA TOTAL': metricas_negocio['COSTO TOTAL ($)'],
        'Mejores Parámetros': grid_search.best_params_
    })
    
    # Guardamos el objeto entrenado por si resulta ser el campeón absoluto
    modelos_entrenados[exp['nombre']] = mejor_modelo

print("====== ¡TORNEO FINALIZADO CON ÉXITO! ======")

====== INICIANDO EL TORNEO DE ALGORITMOS ======

🤖 Entrenando: Regresión Logística (Ruta A - Manual)...
   ✓ ¡Completado! [Mejor F1 en CV: 0.6130] -> [F1 en Validación: 0.6441]
   ✓ Costo Total para la Empresa: $37,040.00 | Tiempo: 4.9s

🤖 Entrenando: SVM (Ruta A - Manual)...
   ✓ ¡Completado! [Mejor F1 en CV: 0.6231] -> [F1 en Validación: 0.6137]
   ✓ Costo Total para la Empresa: $41,420.00 | Tiempo: 3.9s

🤖 Entrenando: Random Forest (Ruta A - Manual)...
   ✓ ¡Completado! [Mejor F1 en CV: 0.6213] -> [F1 en Validación: 0.5549]
   ✓ Costo Total para la Empresa: $46,780.00 | Tiempo: 5.9s

🤖 Entrenando: XGBoost (Ruta A - Manual)...
   ✓ ¡Completado! [Mejor F1 en CV: 0.6134] -> [F1 en Validación: 0.6078]
   ✓ Costo Total para la Empresa: $40,000.00 | Tiempo: 1.7s

🤖 Entrenando: Regresión Logística (Ruta B - ElasticNet)...
   ✓ ¡Completado! [Mejor F1 en CV: 0.6149] -> [F1 en Validación: 0.6293]
   ✓ Costo Total para la Empresa: $39,100.00 | Tiempo: 1.6s

🤖 Entrenando: SVM (Ruta B - Fuerte L

In [24]:
# =========================================================================
# 5. VISUALIZACIÓN DEL RANKING DE RENDIMIENTO DE NEGOCIO
# =========================================================================
df_resultados = pd.DataFrame(resultados_lista)
# Ordenamos de menor pérdida de dinero a mayor pérdida (El que menos dinero pierde, gana)
df_resultados = df_resultados.sort_values(by='PÉRDIDA ECONÓMICA TOTAL', ascending=True).reset_index(drop=True)

# Desplegamos la tabla de posiciones en el notebook
df_resultados

,Modelo / Algoritmo,Estrategia Datos,F1-Score (Validación),Falsos Positivos (Descuentos Inútiles),Falsos Negativos (Fugas No Detectadas),PÉRDIDA ECONÓMICA TOTAL,Mejores Parámetros
0,Regresión Logística (Ruta A - Manual),MANUAL,0.6441,82,118,37040,{'C': 10}
1,Regresión Logística (Ruta B - ElasticNet),FULL,0.6293,80,125,39100,"{'C': 0.1, 'l1_ratio': 0.5}"
2,XGBoost (Ruta B - Regularizado L1/L2),FULL,0.6154,88,127,39860,"{'max_depth': 3, 'n_estimators': 100, 'reg_alp..."
3,XGBoost (Ruta A - Manual),MANUAL,0.6078,95,127,40000,"{'learning_rate': 0.1, 'max_depth': 5, 'n_esti..."
4,SVM (Ruta B - Fuerte L2),FULL,0.6171,73,133,41360,{'C': 1}
5,SVM (Ruta A - Manual),MANUAL,0.6137,76,133,41420,"{'C': 1, 'gamma': 'scale'}"
6,Random Forest (Ruta A - Manual),MANUAL,0.5549,89,150,46780,"{'max_depth': None, 'min_samples_split': 5, 'n..."
7,Random Forest (Ruta B - Full),FULL,0.5495,89,152,47380,"{'max_depth': None, 'min_samples_split': 2, 'n..."


In [27]:
# 1. Recuperamos la Regresión Logística Campeona (Asegúrate que sea la que entrenaste con el SMOTE 30/70)
modelo_campeon = modelos_entrenados['Regresión Logística (Ruta A - Manual)']

# 2. En lugar de predecir 0 o 1, predecimos la PROBABILIDAD de fuga (Clase 1)
y_prob = modelo_campeon.predict_proba(X_val_manual)[:, 1]

# 3. Bucle para probar distintos umbrales
umbrales = np.linspace(0.01, 0.99, 100)
costos = []
f1_scores = []

for umbral in umbrales:
    # Si la prob es mayor al umbral, es 1 (Fuga), sino 0
    y_pred_umbral = (y_prob >= umbral).astype(int)
    
    # Calculamos el costo con tu función
    resultado = calcular_costo_negocio(y_val, y_pred_umbral)
    costos.append(resultado['COSTO TOTAL ($)'])
    
    # Calculamos también el F1-Score para comparar
    f1_scores.append(f1_score(y_val, y_pred_umbral))

# 4. Encontrar el umbral ganador (el que minimiza el costo)
indice_min_costo = np.argmin(costos)
mejor_umbral = umbrales[indice_min_costo]
costo_minimo = costos[indice_min_costo]

print("====== OPTIMIZACIÓN DE UMBRAL (THRESHOLD) ======")
print(f"🔹 Umbral por defecto (0.50): Pérdida de ${calcular_costo_negocio(y_val, (y_prob >= 0.5).astype(int))['COSTO TOTAL ($)']:,.2f}")
print(f"🏆 MEJOR UMBRAL ENCONTRADO: {mejor_umbral:.2f}")
print(f"💰 Nueva Pérdida Mínima: ${costo_minimo:,.2f}")
print(f"📈 F1-Score en ese umbral: {f1_scores[indice_min_costo]:.4f}")
print("================================================")

====== OPTIMIZACIÓN DE UMBRAL (THRESHOLD) ======
🔹 Umbral por defecto (0.50): Pérdida de $37,040.00
🏆 MEJOR UMBRAL ENCONTRADO: 0.05
💰 Nueva Pérdida Mínima: $12,220.00
📈 F1-Score en ese umbral: 0.5047
